# 09 — MongoDB: Replicação e Sharding

**Objetivos:** compreender a arquitetura e distinguir uma instância de laboratório de um cluster distribuído

**Como estudar:** leia a explicação, execute os exemplos, interprete os resultados, refaça os exercícios e consulte o resumo. Exemplos sintéticos são identificados; serviços externos são opcionais e precisam estar disponíveis para laboratórios de infraestrutura.

### Materiais complementares

- [Guia Técnico](./reference_materials/09_mongodb_replicacao_sharding.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_mongodb_replicacao_sharding.pdf)


In [1]:
from pathlib import Path
import pandas as pd
BASE = Path.cwd()
DATA = BASE / 'datasets'
OUT = BASE / 'outputs'
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
assert (DATA / 'leituras_sensores.csv').exists(), 'Abra o JupyterLab na raiz da disciplina (onde está datasets/).'


## Replicação versus sharding

Replica set fornece redundância e eleição; sharding divide dados entre shards. Um `mongod` isolado não demonstra distribuição. `mongos` roteia consultas em um cluster fragmentado.

In [2]:
arquitetura=pd.DataFrame({'componente':['mongod','replica set','mongos','config servers','shard'],'funcao':['servidor de dados','replicação/eleição','roteamento','metadados do cluster','armazena subconjunto de dados']});display(arquitetura)

,componente,funcao
0,mongod,servidor de dados
1,replica set,replicação/eleição
2,mongos,roteamento
3,config servers,metadados do cluster
4,shard,armazena subconjunto de dados


## Shard key e distribuição

A chave deve equilibrar distribuição, cardinalidade, monotonicidade e consultas. Uma chave crescente pode concentrar gravações; a decisão real depende de métricas e padrão de acesso.

In [3]:
cases=pd.DataFrame({'chave':['estado','usuario_id','timestamp'],'risco':['baixa cardinalidade','boa cardinalidade potencial','hotspot se monotônica'],'observacao':['frequentemente insuficiente','ver padrão de consulta','avaliar estratégia hashed']});display(cases)

,chave,risco,observacao
0,estado,baixa cardinalidade,frequentemente insuficiente
1,usuario_id,boa cardinalidade potencial,ver padrão de consulta
2,timestamp,hotspot se monotônica,avaliar estratégia hashed


## Laboratório opcional — pré-condição real

Este curso não simula cluster. Uma instância simples MongoDB de laboratório pode responder ao ping, mas não executará sharding real. O roteiro de cluster multi-nó deve ser configurado especificamente se houver recursos.

In [4]:
# A execução local não ativa conexões automaticamente.
# Para laboratórios opcionais, configure RUN_LIVE_DB=1 antes de iniciar o kernel,
# depois de verificar que a instância é EXCLUSIVAMENTE de estudos.
import os
print('Laboratório externo:', 'habilitado' if os.getenv('RUN_LIVE_DB') == '1' else 'desabilitado (padrão seguro)')

Laboratório externo: desabilitado (padrão seguro)


In [5]:
import os
if os.getenv('RUN_LIVE_DB')=='1':
 from pymongo import MongoClient
 client=MongoClient('mongodb://localhost:27017',serverSelectionTimeoutMS=3000)
 print('Servidor ativo:',client.admin.command('ping'))
 client.close()
 print('Atenção: ping bem-sucedido NÃO comprova cluster fragmentado.')
else:
 print('Teoria e checklist executados; nenhum cluster foi iniciado.')

Teoria e checklist executados; nenhum cluster foi iniciado.


## Exercício resolvido — escolha de chave

Para eventos por usuário e dia, uma chave composta pode refletir filtros frequentes, mas deve ser avaliada contra dispersão de usuários e crescimento. Não fixe um shard key sem caracterizar volume e consulta.

## Consulta Rápida e critérios de revisão

- **Antes de executar:** confirme o diretório de trabalho e as dependências do notebook.
- **Antes de concluir:** separe fatos observados, hipóteses, limitações e resultados sintéticos.
- **Antes de publicar:** não inclua credenciais, dados pessoais, volumes de bancos ou PDFs privados.
- **Exercício adicional:** altere uma hipótese do exemplo e explique o que mudou na saída, com evidência do código.

---
### Aprofundamento v2 — Evidências de topologia real

`ping` prova apenas acesso ao servidor. Para demonstrar **replica set**, é necessário observar estado e membros; para demonstrar **sharding**, é preciso uma topologia com `mongos` e shards. Um servidor único não basta. A simulação abaixo é apenas instrumento conceitual de distribuição de chaves.

**Interpretação:** compare o resultado com a teoria anterior; este exemplo local não substitui um laboratório de serviço externo.

In [6]:
from collections import Counter
chaves=[f'cliente-{i}' for i in range(100)]
baldes=Counter(sum(map(ord,chave))%4 for chave in chaves)
print('Distribuição de brinquedo (NÃO é sharding MongoDB):',dict(sorted(baldes.items())))
assert sum(baldes.values())==100

Distribuição de brinquedo (NÃO é sharding MongoDB): {0: 24, 1: 25, 2: 26, 3: 25}


**Exercício resolvido — revisão:** identifique a principal hipótese do exemplo acima, explique por que ela importa e cite uma condição na qual a técnica falharia. **Resposta orientadora:** registre a pré-condição (tipo de dado, chave, restrição, volume ou topologia), teste-a explicitamente e explique os limites dos resultados observados.

### Resultado reproduzível em `outputs/`

A célula abaixo exporta uma evidência local do estudo. Não conecta a servidores externos.

In [7]:
cases.to_csv(OUT / '09_avaliacao_chaves_sharding.csv',index=False)
print('Exportado: outputs/09_avaliacao_chaves_sharding.csv')

Exportado: outputs/09_avaliacao_chaves_sharding.csv
